In [ ]:
import zipfile, os
from google.colab import drive
drive.mount('/content/drive')

zip_path = '/content/drive/MyDrive/GNSS_csir/IISC.2010.trop.zip'

with zipfile.ZipFile(zip_path, 'r') as z:
    files = sorted(z.namelist())
    print(f"Total files inside: {len(files)}")
    print("First 5:", files[:5])
    print("Last 5:", files[-5:])

    # Read one file to confirm structure
    first_file = files[0]
    print(f"\nReading: {first_file}")
    with z.open(first_file) as f:
        content = f.read().decode('utf-8')
        print(content[:500])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total files inside: 360
First 5: ['IISC.2010.004.trop.gz', 'IISC.2010.005.trop.gz', 'IISC.2010.006.trop.gz', 'IISC.2010.007.trop.gz', 'IISC.2010.008.trop.gz']
Last 5: ['IISC.2010.361.trop.gz', 'IISC.2010.362.trop.gz', 'IISC.2010.363.trop.gz', 'IISC.2010.364.trop.gz', 'IISC.2010.365.trop.gz']

Reading: IISC.2010.004.trop.gz


UnicodeDecodeError: 'utf-8' codec can't decode byte 0x8b in position 1: invalid start byte

In [ ]:
import zipfile, gzip, io

zip_path = '/content/drive/MyDrive/GNSS_csir/IISC.2010.trop.zip'

with zipfile.ZipFile(zip_path, 'r') as z:
    first_file = sorted(z.namelist())[0]
    print(f"Reading: {first_file}")

    with z.open(first_file) as zf:
        with gzip.open(zf, 'rt') as gz:
            for i, line in enumerate(gz):
                print(line, end='')
                if i > 50:
                    break

Reading: IISC.2010.004.trop.gz
%=TRO 1.00 NGL 25:177:51145 NGL 10:004:00000 10:004:86100 P IISC
+FILE/REFERENCE 
 DESCRIPTION    Nevada Geodetic Laboratory, University of Nevada, Reno, USA
 OUTPUT         Total/Wet Zenith Tropo Delay, Gradients, Water Vapor, Mean Temp
 CONTACT        Geoffrey Blewitt; gblewitt@unr.edu; http://geodesy.unr.edu
 SOFTWARE       GipsyX-2.3 gd2e.py w/NGL pppCluster, pppZap, pppTrop, wet2vapor
 HARDWARE       NGL Dell PowerEdge x86_64 GNU/Linux/Fedora custom cluster
 INPUT          RINEX; JPL/IGS20 orbits,IONEX; VMF1/NWM grid; Chalmers/JPL GOT4.8/ac
-FILE/REFERENCE 
*
*
+SITE/ID
*CODE PT DOMES____ T _STATION DESCRIPTION__ APPROX_LON_ APPROX_LAT_ _APP_H_
 IISC  A 22306M002 P Bangalore,India         77 34 13.4  13  1 16.2   844.1
-SITE/ID
*
+SITE/RECEIVER
*SITE PT SOLN T DATA_START__ DATA_END____ DESCRIPTION_________ S/N__ FIRMWARE___
 IISC  A 0001 P 10:004:00000 10:004:86100 ASHTECH UZ-12        ----- -----------
-SITE/RECEIVER
*
+SITE/ANTENNA
*SITE PT SOLN T 

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║   MASTER DATASET CREATION PIPELINE — v3 (FINAL)                     ║
# ║   Sources : NASA POWER (2010-2025) + NGL IISC yearly zips           ║
# ║                                                                      ║
# ║   ACTUAL FILE STRUCTURE IN /content/drive/MyDrive/GNSS_csir:        ║
# ║     IISC.2010.trop.zip  ← zip containing daily .trop.gz files       ║
# ║     IISC.2011.trop.zip                                               ║
# ║     ...                                                              ║
# ║     IISC.2025.trop.zip                                               ║
# ║     POWER_Point_Hourly_20100101_20251231_*.csv                       ║
# ║                                                                      ║
# ║   Inside each zip: IISC.YYYY.DDD.trop.gz (one per day)              ║
# ║   Inside each .gz: SNRG/SINEX troposphere file, 5-min sampling      ║
# ║                                                                      ║
# ║   Run each cell top to bottom in Google Colab                        ║
# ╚══════════════════════════════════════════════════════════════════════╝


# ─────────────────────────────────────────────────────────────────
# CELL 1 — Mount Drive & Imports
# ─────────────────────────────────────────────────────────────────
from google.colab import drive
import os, glob, gzip, zipfile, io
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

print("Mounting Google Drive...")
drive.mount('/content/drive')

PROJECT_FOLDER = '/content/drive/MyDrive/GNSS_csir'
os.chdir(PROJECT_FOLDER)
print(f"Working directory: {os.getcwd()}")

print("\nFiles found in project folder:")
for f in sorted(os.listdir(PROJECT_FOLDER)):
    print(f"  {f}")


# ─────────────────────────────────────────────────────────────────
# CELL 2 — Configuration
# ─────────────────────────────────────────────────────────────────
STATION_ID   = "IISC"
START_YEAR   = 2010
END_YEAR     = 2025
OUTPUT_CSV   = "LSTM_Master_Dataset_2010_2025.csv"

# NASA POWER missing value
NASA_MISSING = -999.0

# NGL physical sanity bounds for Bangalore altitude (~844m)
ZTD_MIN, ZTD_MAX = 1500.0, 3500.0   # mm
PWV_MIN, PWV_MAX = 0.0,   100.0     # mm


# ─────────────────────────────────────────────────────────────────
# CELL 3 — Parse ALL NGL Files
#           Structure: IISC.YYYY.trop.zip → IISC.YYYY.DDD.trop.gz
# ─────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("  STEP 1: Parsing NGL IISC Troposphere Files")
print("="*60)

satellite_records = []
total_zips        = 0
total_gz_files    = 0
total_gz_failed   = 0
parse_errors      = 0

for year in range(START_YEAR, END_YEAR + 1):

    # Expected zip filename pattern: IISC.2010.trop.zip
    zip_path = os.path.join(PROJECT_FOLDER, f"IISC.{year}.trop.zip")

    if not os.path.exists(zip_path):
        print(f"  WARNING: IISC.{year}.trop.zip not found — skipping {year}")
        continue

    total_zips  += 1
    year_records = 0

    try:
        with zipfile.ZipFile(zip_path, 'r') as zf:
            # Get all .gz files inside this yearly zip
            gz_names = sorted([n for n in zf.namelist() if n.endswith('.gz')])
            total_gz_files += len(gz_names)

            for gz_name in gz_names:
                try:
                    # Double decompression: unzip → ungzip → text
                    with zf.open(gz_name) as zipped_gz:
                        with gzip.open(zipped_gz, 'rt') as gz:
                            in_solution = False

                            for line in gz:
                                # Enter data block
                                if '+TROP/SOLUTION' in line:
                                    in_solution = True
                                    continue
                                # Exit data block
                                if '-TROP/SOLUTION' in line:
                                    in_solution = False
                                    continue
                                if not in_solution:
                                    continue
                                # Only process IISC data lines
                                if not line.startswith(f' {STATION_ID}'):
                                    continue
                                if ':' not in line:
                                    continue

                                parts = line.split()
                                if len(parts) < 12:
                                    continue

                                try:
                                    epoch = parts[1]         # YY:DOY:SSSSS
                                    ztd   = float(parts[2])  # TROTOT mm
                                    pwv   = float(parts[9])  # WVAPOR mm

                                    # Sanity check physical bounds
                                    if not (ZTD_MIN < ztd < ZTD_MAX):
                                        continue
                                    if not (PWV_MIN <= pwv <= PWV_MAX):
                                        continue

                                    # Parse epoch YY:DOY:SSSSS → datetime
                                    yy, doy, sec = epoch.split(':')
                                    yr   = int(yy) + 2000
                                    base = datetime(yr, 1, 1) + timedelta(days=int(doy) - 1)
                                    dt   = base + timedelta(seconds=int(sec))

                                    satellite_records.append({
                                        'Datetime': dt,
                                        'ZTD': ztd,
                                        'PWV': pwv
                                    })
                                    year_records += 1

                                except (ValueError, IndexError):
                                    parse_errors += 1
                                    continue

                except Exception:
                    total_gz_failed += 1
                    continue

    except zipfile.BadZipFile:
        print(f"  ERROR: IISC.{year}.trop.zip is corrupted — skipping")
        continue

    print(f"  {year}: {len(gz_names):>3} day-files | {year_records:>7,} 5-min records")

print(f"\n  Yearly zips processed   : {total_zips}")
print(f"  Daily .gz files read    : {total_gz_files:,}")
print(f"  Daily .gz files failed  : {total_gz_failed:,}")
print(f"  Total 5-min records     : {len(satellite_records):,}")
print(f"  Parse errors skipped    : {parse_errors:,}")


# ─────────────────────────────────────────────────────────────────
# CELL 4 — Resample NGL from 5-min to Hourly
# ─────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("  STEP 2: Resampling NGL 5-min → Hourly Mean")
print("="*60)

df_sat = pd.DataFrame(satellite_records)
df_sat['Datetime'] = pd.to_datetime(df_sat['Datetime'])
df_sat = df_sat.sort_values('Datetime')

# Remove exact duplicate timestamps
dupes = df_sat.duplicated(subset='Datetime').sum()
df_sat = df_sat.drop_duplicates(subset='Datetime')
print(f"  Duplicate 5-min records removed : {dupes:,}")

# Resample to hourly mean (12 x 5-min observations per hour)
df_sat.set_index('Datetime', inplace=True)
df_sat_hourly = df_sat.resample('1H').mean()

# Drop hours that had zero NGL observations
empty_hours   = df_sat_hourly.isna().all(axis=1).sum()
df_sat_hourly = df_sat_hourly.dropna().reset_index()

print(f"  Hourly NGL records      : {len(df_sat_hourly):,}")
print(f"  Empty hours dropped     : {empty_hours:,}")
print(f"  NGL date range          : {df_sat_hourly['Datetime'].min()} → {df_sat_hourly['Datetime'].max()}")
print(f"  ZTD range               : {df_sat_hourly['ZTD'].min():.1f} – {df_sat_hourly['ZTD'].max():.1f} mm")
print(f"  PWV range               : {df_sat_hourly['PWV'].min():.2f} – {df_sat_hourly['PWV'].max():.2f} mm")


# ─────────────────────────────────────────────────────────────────
# CELL 5 — Load & Clean NASA POWER CSV
# ─────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("  STEP 3: Loading NASA POWER CSV")
print("="*60)

# Auto-detect NASA POWER file
nasa_files = glob.glob(os.path.join(PROJECT_FOLDER, "POWER_*.csv"))
if not nasa_files:
    raise FileNotFoundError(
        "No NASA POWER CSV found. Filename must start with 'POWER_'"
    )
nasa_path = nasa_files[0]
print(f"  File: {os.path.basename(nasa_path)}")

# Skip the 12-line metadata header
df_nasa = pd.read_csv(nasa_path, skiprows=12)
print(f"  Raw rows loaded         : {len(df_nasa):,}")
print(f"  Columns                 : {list(df_nasa.columns)}")

# Replace -999 missing value indicator
missing_count = (df_nasa == NASA_MISSING).sum().sum()
df_nasa.replace(NASA_MISSING, np.nan, inplace=True)
print(f"  Missing (-999) replaced : {missing_count:,}")

# Build Datetime from YEAR, MO, DY, HR columns
df_nasa['Datetime'] = pd.to_datetime(
    df_nasa[['YEAR', 'MO', 'DY', 'HR']].rename(
        columns={'YEAR': 'year', 'MO': 'month', 'DY': 'day', 'HR': 'hour'}
    )
)

# Remove any duplicate timestamps
dupes_nasa = df_nasa.duplicated(subset='Datetime').sum()
df_nasa = df_nasa.drop_duplicates(subset='Datetime').sort_values('Datetime')
print(f"  Duplicate rows removed  : {dupes_nasa:,}")

# ── CRITICAL: PRECTOTCORR is mm/day → convert to mm/hour
df_nasa['PRECTOTCORR'] = df_nasa['PRECTOTCORR'] / 24.0

# Keep and rename only needed columns
df_nasa = df_nasa[['Datetime', 'T2M', 'RH2M', 'PS', 'PRECTOTCORR']]
df_nasa.rename(columns={
    'T2M'         : 'Temperature_C',
    'RH2M'        : 'Relative_Humidity',
    'PS'          : 'Surface_Pressure_kPa',
    'PRECTOTCORR' : 'Precipitation_mm'
}, inplace=True)

print(f"  Final NASA rows         : {len(df_nasa):,}")
print(f"  Date range              : {df_nasa['Datetime'].min()} → {df_nasa['Datetime'].max()}")
print(f"  Temperature_C           : {df_nasa['Temperature_C'].min():.1f} – {df_nasa['Temperature_C'].max():.1f} °C")
print(f"  Relative_Humidity       : {df_nasa['Relative_Humidity'].min():.1f} – {df_nasa['Relative_Humidity'].max():.1f} %")
print(f"  Surface_Pressure_kPa    : {df_nasa['Surface_Pressure_kPa'].min():.2f} – {df_nasa['Surface_Pressure_kPa'].max():.2f}")
print(f"  Precipitation mm/hr     : {df_nasa['Precipitation_mm'].min():.5f} – {df_nasa['Precipitation_mm'].max():.4f}")


# ─────────────────────────────────────────────────────────────────
# CELL 6 — Merge NASA + NGL on Datetime
# ─────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("  STEP 4: Merging NASA POWER + NGL on Datetime")
print("="*60)

master_df = pd.merge(df_nasa, df_sat_hourly, on='Datetime', how='inner')
master_df = master_df.sort_values('Datetime').reset_index(drop=True)

print(f"  NASA rows               : {len(df_nasa):,}")
print(f"  NGL hourly rows         : {len(df_sat_hourly):,}")
print(f"  Merged rows             : {len(master_df):,}")
print(f"  Rows lost in merge      : {len(df_nasa) - len(master_df):,}  ← hours NGL had no data")
print(f"  Date range              : {master_df['Datetime'].min()} → {master_df['Datetime'].max()}")


# ─────────────────────────────────────────────────────────────────
# CELL 7 — Missing Value Handling
# ─────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("  STEP 5: Missing Value Handling")
print("="*60)

print(f"  Missing per column BEFORE handling:")
print(master_df.isna().sum().to_string())

feature_cols = ['Temperature_C', 'Relative_Humidity',
                'Surface_Pressure_kPa', 'ZTD', 'PWV', 'Precipitation_mm']

# Reindex to full continuous hourly timeline to expose hidden gaps
master_df.set_index('Datetime', inplace=True)
full_index = pd.date_range(
    start=master_df.index.min(),
    end=master_df.index.max(),
    freq='1H'
)
master_df        = master_df.reindex(full_index)
master_df.index.name = 'Datetime'
hidden_gaps      = master_df.isna().any(axis=1).sum()
print(f"\n  Hidden gaps after reindex : {hidden_gaps:,} hours")

# Short gaps ≤ 3h → linear interpolation (sensor dropout)
master_df[feature_cols] = master_df[feature_cols].interpolate(
    method='linear', limit=3, limit_direction='forward'
)

# Medium gaps ≤ 24h → forward fill
master_df[feature_cols] = master_df[feature_cols].ffill(limit=24)

# Precipitation gaps → 0 (missing most likely means no rain)
master_df['Precipitation_mm'] = master_df['Precipitation_mm'].fillna(0.0)

# Drop anything still missing (gaps > 24h — too unreliable to fill)
rows_before  = len(master_df)
master_df.dropna(inplace=True)
rows_dropped = rows_before - len(master_df)

print(f"  Rows dropped (gaps >24h) : {rows_dropped:,}")
print(f"  Missing values after     : {master_df.isna().sum().sum()}")
print(f"  Rows remaining           : {len(master_df):,}")

master_df.reset_index(inplace=True)


# ─────────────────────────────────────────────────────────────────
# CELL 8 — Add Cyclic Time + Precipitation Lag Features
# ─────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("  STEP 6: Adding Cyclic Time + Lag Features")
print("="*60)

# Cyclic time encoding — converts hour/month to continuous circular values
master_df['hour_sin']  = np.sin(2 * np.pi * master_df['Datetime'].dt.hour / 24)
master_df['hour_cos']  = np.cos(2 * np.pi * master_df['Datetime'].dt.hour / 24)
master_df['month_sin'] = np.sin(2 * np.pi * master_df['Datetime'].dt.month / 12)
master_df['month_cos'] = np.cos(2 * np.pi * master_df['Datetime'].dt.month / 12)

# Precipitation lag features
# These were missing in previous training and caused XGBoost to win
master_df['Precipitation_Lag1']  = master_df['Precipitation_mm'].shift(1)
master_df['Precipitation_Lag3']  = master_df['Precipitation_mm'].shift(3)
master_df['Precipitation_Lag6']  = master_df['Precipitation_mm'].shift(6)
master_df['Precipitation_Lag24'] = master_df['Precipitation_mm'].shift(24)

# Drop first 24 rows that now have NaN lags
master_df.dropna(inplace=True)
master_df.reset_index(drop=True, inplace=True)

print(f"  Cyclic features added : hour_sin, hour_cos, month_sin, month_cos")
print(f"  Lag features added    : Lag1, Lag3, Lag6, Lag24")
print(f"  Total columns         : {len(master_df.columns)}")
print(f"\n  Final column list:")
for i, col in enumerate(master_df.columns, 1):
    print(f"    {i:>2}. {col}")


# ─────────────────────────────────────────────────────────────────
# CELL 9 — Final Data Quality Report
# ─────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("  FINAL DATA QUALITY REPORT")
print("="*60)

all_feat_cols = [c for c in master_df.columns if c != 'Datetime']

print(f"\n  Date range     : {master_df['Datetime'].min()} → {master_df['Datetime'].max()}")
print(f"  Total rows     : {len(master_df):,}")
total_possible = (END_YEAR - START_YEAR + 1) * 365 * 24
print(f"  Coverage       : {100*len(master_df)/total_possible:.1f}% of 15 years")
print(f"  Missing values : {master_df[all_feat_cols].isna().sum().sum()}")

# Yearly coverage check
print(f"\n  Yearly coverage:")
master_df['_yr'] = master_df['Datetime'].dt.year
for yr, cnt in master_df.groupby('_yr').size().items():
    expected = 8784 if yr % 4 == 0 else 8760
    pct      = 100 * cnt / expected
    flag     = "✓" if pct > 90 else "⚠  LOW — check NGL zip for this year"
    print(f"    {yr}: {cnt:>6,} hrs  ({pct:.1f}%)  {flag}")
master_df.drop(columns=['_yr'], inplace=True)

# Precipitation distribution
y = master_df['Precipitation_mm']
print(f"\n  Precipitation (mm/hr):")
print(f"    Zero hours      : {(y==0).sum():,}  ({100*(y==0).mean():.1f}%)")
print(f"    Non-zero hours  : {(y>0).sum():,}  ({100*(y>0).mean():.1f}%)")
print(f"    Max             : {y.max():.4f} mm/hr")
print(f"    P99             : {np.percentile(y,99):.4f} mm/hr")
print(f"    Mean (non-zero) : {y[y>0].mean():.4f} mm/hr")

print(f"\n  Core feature statistics:")
print(master_df[['Temperature_C', 'Relative_Humidity',
                  'Surface_Pressure_kPa', 'ZTD', 'PWV',
                  'Precipitation_mm']].describe().round(3).to_string())

print("\n" + "="*60)


# ─────────────────────────────────────────────────────────────────
# CELL 10 — Save Master Dataset
# ─────────────────────────────────────────────────────────────────
output_path = os.path.join(PROJECT_FOLDER, OUTPUT_CSV)
master_df.to_csv(output_path, index=False)

print(f"\n  Saved → {OUTPUT_CSV}")
print(f"  Rows    : {len(master_df):,}")
print(f"  Columns : {len(master_df.columns)}")
print(f"\n  First 3 rows:")
print(master_df.head(3).to_string())
print(f"\n  Pipeline complete — ready for sequence preparation!")

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/MyDrive/GNSS_csir

Files found in project folder:
  Data_merging
  IISC.2010.trop.zip
  IISC.2011.trop.zip
  IISC.2012.trop.zip
  IISC.2013.trop.zip
  IISC.2014.trop.zip
  IISC.2015.trop.zip
  IISC.2016.trop.zip
  IISC.2017.trop.zip
  IISC.2018.trop.zip
  IISC.2019.trop.zip
  IISC.2020.trop.zip
  IISC.2021.trop.zip
  IISC.2022.trop.zip
  IISC.2023.trop.zip
  IISC.2024.trop.zip
  IISC.2025.trop.zip
  IISC_Raw_Data
  Model_1_XGBoost_Prep.ipynb
  POWER_Point_Hourly_20100101_20251231_013d01N_077d56E_LST.csv
  Two_stage_cnn_lstm
  Untitled0.ipynb
  XGBoost_Training_Dataset.csv
  cnn_lstm_hybrid_with_self_attention
  cnn_lstm_peak_hunter
  cnn_lstm_peak_hunterv2_with_reduced_panalty_to3x
  logs_bangalore
  lstm
  lstm_data_prep_for_phase2.ipynb
  phase1_lstm_data
  phase2_lstm_data
  using_transformers

  S

/tmp/ipykernel_8572/202209183.py:179: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_sat_hourly = df_sat.resample('1H').mean()


  Raw rows loaded         : 140,256
  Columns                 : ['YEAR', 'MO', 'DY', 'HR', 'T2M', 'RH2M', 'PRECTOTCORR', 'PS']
  Missing (-999) replaced : 0
  Duplicate rows removed  : 0
  Final NASA rows         : 140,256
  Date range              : 2010-01-01 00:00:00 → 2025-12-31 23:00:00
  Temperature_C           : 9.2 – 40.8 °C
  Relative_Humidity       : 3.3 – 100.0 %
  Surface_Pressure_kPa    : 90.12 – 92.65
  Precipitation mm/hr     : 0.00000 – 30.6108

  STEP 4: Merging NASA POWER + NGL on Datetime
  NASA rows               : 140,256
  NGL hourly rows         : 133,034
  Merged rows             : 133,034
  Rows lost in merge      : 7,222  ← hours NGL had no data
  Date range              : 2010-01-04 02:00:00 → 2025-12-31 23:00:00

  STEP 5: Missing Value Handling
  Missing per column BEFORE handling:
Datetime                0
Temperature_C           0
Relative_Humidity       0
Surface_Pressure_kPa    0
Precipitation_mm        0
ZTD                     0
PWV                   

/tmp/ipykernel_8572/202209183.py:282: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  full_index = pd.date_range(


  Rows dropped (gaps >24h) : 5,443
  Missing values after     : 0
  Rows remaining           : 134,739

  STEP 6: Adding Cyclic Time + Lag Features
  Cyclic features added : hour_sin, hour_cos, month_sin, month_cos
  Lag features added    : Lag1, Lag3, Lag6, Lag24
  Total columns         : 15

  Final column list:
     1. Datetime
     2. Temperature_C
     3. Relative_Humidity
     4. Surface_Pressure_kPa
     5. Precipitation_mm
     6. ZTD
     7. PWV
     8. hour_sin
     9. hour_cos
    10. month_sin
    11. month_cos
    12. Precipitation_Lag1
    13. Precipitation_Lag3
    14. Precipitation_Lag6
    15. Precipitation_Lag24

  FINAL DATA QUALITY REPORT

  Date range     : 2010-01-05 02:00:00 → 2025-12-31 23:00:00
  Total rows     : 134,715
  Coverage       : 96.1% of 15 years
  Missing values : 0

  Yearly coverage:
    2010:  8,649 hrs  (98.7%)  ✓
    2011:  7,994 hrs  (91.3%)  ✓
    2012:  8,775 hrs  (99.9%)  ✓
    2013:  7,683 hrs  (87.7%)  ⚠  LOW — check NGL zip for this year